In [1]:
import json
import joblib
import numpy as np
import os
import pandas as pd     
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, cohen_kappa_score


#import all algo
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import SGDClassifier

In [71]:
#load the intent.json file
with open('intents.json') as f:
    data = json.load(f)

#store the patterns and tags in separate lists
#the patterns are the sentences that the user will input 
# and the tags are the labels that we want to predict
sentences = []
labels = []
for intent in data['intents']:
   
    for pattern in intent['patterns']:
        sentences.append(pattern)
        labels.append(intent['tag'])


print("total sentences: ", len(sentences))
print("total labels: ", len(labels))

total sentences:  96
total labels:  96


In [75]:
#train test split
#stratify is used to ensure that the distribution of labels in the train and test sets is the same as in the original dataset
#not using stratify will distribute the labels randomly and may lead to an imbalanced distribution of..... 
# labels in the train and test sets, which can affect the performance of the model
x_train, x_test, y_train, y_test = train_test_split(sentences, labels, test_size=0.2, random_state=42, stratify=labels)


print(f"Train text samples: {len(x_train)}")
print(f"Test text samples: {len(x_test)}")
print()


Train text samples: 76
Test text samples: 20



In [86]:
vectorizer = TfidfVectorizer()
x_train = vectorizer.fit_transform(x_train)
x_test = vectorizer.transform(x_test)
print("trainin data shape: ", x_train.shape)
print("testing data shape: ", x_test.shape)


trainin data shape:  (76, 136)
testing data shape:  (20, 136)


In [87]:
#eveluate the model using different algorithms and compare their performance
def evelute_func(y_test,y_pred):
    acc=accuracy_score(y_test,y_pred)
    print("accurcy",acc)
    cm=confusion_matrix(y_test,y_pred)
    print("confusion metrx",cm)
    kp=cohen_kappa_score(y_test,y_pred)
    print("Kappa scor",kp)
    cr=classification_report(y_test,y_pred)
    print("classification report",cr)
    err=1-acc
    print("error rate",err)

In [88]:
# 1. Multinomial Naive Bayes
nb = MultinomialNB()
nb.fit(x_train, y_train)
y_pred = nb.predict(x_test)
evelute_func(y_test,y_pred)

accurcy 0.5
confusion metrx [[2 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 1 0 0 0 0]
 [2 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 1 0]
 [1 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 2 0 0 0 0]
 [2 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 2 0 0]
 [0 0 0 0 0 0 0 0 2 0]
 [0 0 0 0 0 1 0 1 0 0]]
Kappa scor 0.4444444444444444
classification report                    precision    recall  f1-score   support

     crop_disease       0.29      1.00      0.44         2
       fertilizer       0.50      0.50      0.50         2
          goodbye       0.00      0.00      0.00         2
government_scheme       1.00      0.50      0.67         2
         greeting       0.00      0.00      0.00         2
       irrigation       0.50      1.00      0.67         2
             pest       0.00      0.00      0.00         2
             soil       0.67      1.00      0.80         2
      sowing_time       0.67      1.00      0.80         2
          weather       0.00      0.00      0.00         2

         accuracy                     

In [89]:
#logestic regression
lr = LogisticRegression()
lr.fit(x_train, y_train)
y_pred_lr = lr.predict(x_test)
evelute_func(y_test,y_pred_lr)  

accurcy 0.5
confusion metrx [[1 0 0 0 1 0 0 0 0 0]
 [0 1 0 0 0 1 0 0 0 0]
 [0 0 0 2 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 1 0]
 [0 0 0 1 1 0 0 0 0 0]
 [0 0 0 0 0 2 0 0 0 0]
 [2 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 2 0 0]
 [0 0 0 0 0 0 0 0 2 0]
 [1 0 0 0 0 0 1 0 0 0]]
Kappa scor 0.4444444444444444
classification report                    precision    recall  f1-score   support

     crop_disease       0.25      0.50      0.33         2
       fertilizer       1.00      0.50      0.67         2
          goodbye       0.00      0.00      0.00         2
government_scheme       0.25      0.50      0.33         2
         greeting       0.50      0.50      0.50         2
       irrigation       0.67      1.00      0.80         2
             pest       0.00      0.00      0.00         2
             soil       1.00      1.00      1.00         2
      sowing_time       0.67      1.00      0.80         2
          weather       0.00      0.00      0.00         2

         accuracy                     

In [90]:
#svm
svm = SVC(kernel='linear')
svm.fit(x_train, y_train)
y_pred_svm = svm.predict(x_test)
evelute_func(y_test,y_pred_svm)

accurcy 0.5
confusion metrx [[1 0 0 0 1 0 0 0 0 0]
 [0 1 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 2 0 0 0]
 [0 0 0 1 0 0 0 0 0 1]
 [0 0 0 0 1 0 1 0 0 0]
 [0 0 0 0 0 2 0 0 0 0]
 [2 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 2 0 0]
 [0 0 0 0 0 0 0 0 2 0]
 [1 0 0 0 0 0 1 0 0 0]]
Kappa scor 0.44444444444444464
classification report                    precision    recall  f1-score   support

     crop_disease       0.25      0.50      0.33         2
       fertilizer       1.00      0.50      0.67         2
          goodbye       0.00      0.00      0.00         2
government_scheme       1.00      0.50      0.67         2
         greeting       0.50      0.50      0.50         2
       irrigation       1.00      1.00      1.00         2
             pest       0.00      0.00      0.00         2
             soil       1.00      1.00      1.00         2
      sowing_time       1.00      1.00      1.00         2
          weather       0.00      0.00      0.00         2

         accuracy                    

In [91]:
#random forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(x_train, y_train)
y_pred_rf = rf.predict(x_test)
evelute_func(y_test,y_pred_rf)

accurcy 0.6
confusion metrx [[1 0 0 0 1 0 0 0 0 0]
 [0 1 0 0 1 0 0 0 0 0]
 [0 0 0 0 2 0 0 0 0 0]
 [0 0 0 1 1 0 0 0 0 0]
 [0 0 0 0 2 0 0 0 0 0]
 [0 0 0 0 0 2 0 0 0 0]
 [1 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 2 0 0]
 [0 0 0 0 0 0 0 0 2 0]
 [0 0 0 1 1 0 0 0 0 0]]
Kappa scor 0.5555555555555556
classification report                    precision    recall  f1-score   support

     crop_disease       0.50      0.50      0.50         2
       fertilizer       1.00      0.50      0.67         2
          goodbye       0.00      0.00      0.00         2
government_scheme       0.50      0.50      0.50         2
         greeting       0.25      1.00      0.40         2
       irrigation       1.00      1.00      1.00         2
             pest       1.00      0.50      0.67         2
             soil       1.00      1.00      1.00         2
      sowing_time       1.00      1.00      1.00         2
          weather       0.00      0.00      0.00         2

         accuracy                     

In [96]:
#use k cross validation to find the best k for KNN for get the best performance of KNN
from sklearn.model_selection import cross_val_score

for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, x_train, y_train, cv=5)
    print(k, scores.mean())

    #take the highest score and use that k to train the KNN model and evaluate it
    #here k=6 is the best k for KNN 0.4608....
#even k=6 is best we use k=7 because k=6 is even and it may lead to tie in the voting of KNN and we want to avoid that tie in voting
knn = KNeighborsClassifier(n_neighbors=9)
knn.fit(x_train, y_train)
y_pred_knn = knn.predict(x_test)
evelute_func(y_test,y_pred_knn)

1 0.4608333333333333
2 0.4216666666666667
3 0.4216666666666667
4 0.45999999999999996
5 0.4608333333333333
6 0.48666666666666664
7 0.49916666666666665
8 0.5125
9 0.5
10 0.5
11 0.4875
12 0.4741666666666666
13 0.4608333333333333
14 0.4608333333333333
15 0.45999999999999996
16 0.4466666666666666
17 0.4333333333333333
18 0.4466666666666666
19 0.4208333333333333
20 0.40750000000000003
accurcy 0.5
confusion metrx [[2 0 0 0 0 0 0 0 0 0]
 [0 2 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 2 0 0 0]
 [0 0 0 1 0 0 1 0 0 0]
 [0 0 0 0 0 0 2 0 0 0]
 [0 0 0 0 0 2 0 0 0 0]
 [2 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 2 0 0]
 [0 1 0 0 0 0 0 0 1 0]
 [1 0 0 0 0 1 0 0 0 0]]
Kappa scor 0.4444444444444444
classification report                    precision    recall  f1-score   support

     crop_disease       0.40      1.00      0.57         2
       fertilizer       0.67      1.00      0.80         2
          goodbye       0.00      0.00      0.00         2
government_scheme       1.00      0.50      0.67         2
         gr

In [93]:
#decision tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(x_train, y_train)
y_pred_dt = dt.predict(x_test)
evelute_func(y_test,y_pred_dt)

accurcy 0.6
confusion metrx [[2 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 1 0]
 [0 0 0 0 2 0 0 0 0 0]
 [0 0 0 1 1 0 0 0 0 0]
 [0 0 0 0 2 0 0 0 0 0]
 [0 0 0 0 0 2 0 0 0 0]
 [0 0 0 0 1 0 1 0 0 0]
 [0 0 0 0 0 0 0 2 0 0]
 [1 0 0 0 0 0 0 0 1 0]
 [0 0 1 1 0 0 0 0 0 0]]
Kappa scor 0.5555555555555556
classification report                    precision    recall  f1-score   support

     crop_disease       0.67      1.00      0.80         2
       fertilizer       1.00      0.50      0.67         2
          goodbye       0.00      0.00      0.00         2
government_scheme       0.50      0.50      0.50         2
         greeting       0.33      1.00      0.50         2
       irrigation       1.00      1.00      1.00         2
             pest       1.00      0.50      0.67         2
             soil       1.00      1.00      1.00         2
      sowing_time       0.50      0.50      0.50         2
          weather       0.00      0.00      0.00         2

         accuracy                     

In [94]:
#gradient boosting
gb = GradientBoostingClassifier(random_state=42)
gb.fit(x_train, y_train)
y_pred_gb = gb.predict(x_test)
evelute_func(y_test,y_pred_gb)

accurcy 0.45
confusion metrx [[0 0 0 0 2 0 0 0 0 0]
 [0 2 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 2 0 0 0]
 [0 0 0 0 0 0 2 0 0 0]
 [0 0 0 0 1 0 1 0 0 0]
 [0 0 0 0 0 2 0 0 0 0]
 [1 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 2 0 0]
 [0 0 1 0 0 0 0 0 1 0]
 [1 0 0 0 0 0 1 0 0 0]]
Kappa scor 0.38888888888888884
classification report                    precision    recall  f1-score   support

     crop_disease       0.00      0.00      0.00         2
       fertilizer       1.00      1.00      1.00         2
          goodbye       0.00      0.00      0.00         2
government_scheme       0.00      0.00      0.00         2
         greeting       0.33      0.50      0.40         2
       irrigation       1.00      1.00      1.00         2
             pest       0.14      0.50      0.22         2
             soil       1.00      1.00      1.00         2
      sowing_time       1.00      0.50      0.67         2
          weather       0.00      0.00      0.00         2

         accuracy                   

In [95]:
#sgd
sgd = SGDClassifier(random_state=42)
sgd.fit(x_train, y_train)
y_pred_sgd = sgd.predict(x_test)    
evelute_func(y_test,y_pred_sgd)


accurcy 0.4
confusion metrx [[0 0 0 0 2 0 0 0 0 0]
 [0 1 0 0 0 1 0 0 0 0]
 [0 0 0 2 0 0 0 0 0 0]
 [0 0 0 1 0 1 0 0 0 0]
 [0 0 0 1 1 0 0 0 0 0]
 [0 0 0 0 0 2 0 0 0 0]
 [1 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 2 0 0]
 [0 0 1 0 0 0 0 0 1 0]
 [1 1 0 0 0 0 0 0 0 0]]
Kappa scor 0.3333333333333335
classification report                    precision    recall  f1-score   support

     crop_disease       0.00      0.00      0.00         2
       fertilizer       0.50      0.50      0.50         2
          goodbye       0.00      0.00      0.00         2
government_scheme       0.25      0.50      0.33         2
         greeting       0.33      0.50      0.40         2
       irrigation       0.50      1.00      0.67         2
             pest       0.00      0.00      0.00         2
             soil       1.00      1.00      1.00         2
      sowing_time       0.50      0.50      0.50         2
          weather       0.00      0.00      0.00         2

         accuracy                     